# 图同构特征实验

本示例通过九章 SDK 的本地应用接口加载 MUTAG 图数据，使用采样数据构造事件特征，并训练一个线性分类器观察图之间的可分性。

## 1. 初始化环境

In [ ]:
import numpy as np

from jiuzhang.local.applications import (
    draw_graph,
    event_feature_vector,
    event_feature_vector_from_samples,
    load_graph_samples,
    load_mutag_graphs,
    orbit_feature_vector_from_samples,
    sample_to_event,
    sample_to_orbit,
    train_linear_graph_classifier,
)

import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Sarasa UI SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 2. 加载示例图和采样数据

In [ ]:
graphs = load_mutag_graphs()
sample_sets = load_graph_samples(['samples0.npy', 'samples1.npy', 'samples2.npy', 'samples3.npy'])

print('Graph count:', len(graphs))
print('First graph shape:', graphs[0].shape)
print('First sample:', sample_sets[0][0].tolist())

## 3. 查看图结构

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
for index, ax in enumerate(axes.ravel()):
    draw_graph(graphs[index], title=f'MUTAG graph {index}', ax=ax)
plt.tight_layout()
plt.show()

## 4. 样本到轨道和事件

`sample_to_orbit` 用于得到样本的占据数结构；`sample_to_event` 将样本映射到指定最大光子数下的事件编号。

In [ ]:
example_sample = sample_sets[0][0]
print('Orbit:', sample_to_orbit(example_sample))
print('Event:', sample_to_event(example_sample, max_count=3))

## 5. 构造事件特征

In [ ]:
EVENTS = [8, 10]
MAX_COUNT = 2
features = np.array([
    event_feature_vector_from_samples(samples, EVENTS, MAX_COUNT)
    for samples in sample_sets
])
labels = np.array([1, 0, 0, 1])

print(features)

## 6. 轨道特征和直接图特征

In [ ]:
orbit_features = orbit_feature_vector_from_samples(
    sample_sets[0],
    [[1, 1], [2], [1, 1, 1, 1], [2, 1, 1]],
)
direct_features = event_feature_vector(graphs[0], [2, 4], MAX_COUNT, samples=200, mean_photon_count=5.5)

print('Orbit features:', orbit_features)
print('Direct graph features:', direct_features)

## 7. 训练线性分类器并可视化

In [ ]:
classification = train_linear_graph_classifier(features, labels)

plt.figure(figsize=(6, 4))
for class_id in sorted(set(classification.labels.tolist())):
    mask = classification.labels == class_id
    plt.scatter(
        classification.scaled_features[mask, 0],
        classification.scaled_features[mask, 1],
        label=f'Class {class_id}',
    )
plt.plot(classification.line_x, classification.line_y, color='black', label='Decision boundary')
plt.xlabel('Event 8 feature')
plt.ylabel('Event 10 feature')
plt.legend()
plt.tight_layout()
plt.show()